# Homework GraphFrames Part 1


In [1]:
import sys
sys.path.insert(0, "/home/hadoop/homework/spark-graphframe")

from pyspark.sql import functions as F

from spark_graphframe_homework import create_spark_session, ensure_data, build_graphframes


In [2]:
paths = ensure_data(sync_hdfs=True)
spark = create_spark_session("graphframes-homework-part1")

paths


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/hadoop/hadoop/share/hadoop/common/lib/slf4j-log4j12-1.7.25.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/hadoop/hive/lib/log4j-slf4j-impl-2.10.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.slf4j.impl.Log4jLoggerFactory]


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/hadoop/hadoop/share/hadoop/common/lib/slf4j-log4j12-1.7.25.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/hadoop/hive/lib/log4j-slf4j-impl-2.10.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.slf4j.impl.Log4jLoggerFactory]


2026-03-17 08:17:05,050 INFO  [Thread-7] sasl.SaslDataTransferClient (SaslDataTransferClient.java:checkTrustAndSend(239)) - SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/hadoop/hadoop/share/hadoop/common/lib/slf4j-log4j12-1.7.25.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/hadoop/hive/lib/log4j-slf4j-impl-2.10.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.slf4j.impl.Log4jLoggerFactory]


2026-03-17 08:17:17,713 INFO  [Thread-7] sasl.SaslDataTransferClient (SaslDataTransferClient.java:checkTrustAndSend(239)) - SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false


26/03/17 08:17:20 WARN Utils: Your hostname, bigdata resolves to a loopback address: 127.0.1.1; using 10.3.134.62 instead (on interface ens3)
26/03/17 08:17:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


26/03/17 08:17:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


{'local': {'station_csv': '/home/hadoop/homework/spark-graphframe/data/station.csv',
  'trip_csv': '/home/hadoop/homework/spark-graphframe/data/trip.csv'},
 'hdfs': {'station_csv': 'hdfs://localhost:9000/user/hadoop/graph_sf_by/station.csv',
  'trip_csv': 'hdfs://localhost:9000/user/hadoop/graph_sf_by/trip.csv'}}

In [3]:
graph_payload = build_graphframes(spark, source=paths["local"]["station_csv"])

station_df = graph_payload["station_df"]
trip_df = graph_payload["trip_df"]
vertices = graph_payload["vertices"]
trip_edges = graph_payload["trip_edges"]
route_edges = graph_payload["route_edges"]
trip_graph = graph_payload["trip_graph"]
route_graph = graph_payload["route_graph"]

{
    "stations": station_df.count(),
    "trips": trip_df.count(),
    "trip_edges": trip_edges.count(),
    "route_edges": route_edges.count(),
}



[Stage 0:>                                                          (0 + 1) / 1]



/home/hadoop/homework/.venv/lib/python3.10/site-packages/pyspark/sql/dataframe.py:148: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(



[Stage 8:======================>                                    (3 + 4) / 8]




[Stage 22:=============================>                            (4 + 4) / 8]



{'stations': 70, 'trips': 354152, 'trip_edges': 354152, 'route_edges': 1692}

## Question 1

Find the indegree and outdegree of all stations.


In [4]:
degree_df = (
    vertices.select("id", "name")
    .join(trip_graph.inDegrees, on="id", how="left")
    .join(trip_graph.outDegrees, on="id", how="left")
    .fillna({"inDegree": 0, "outDegree": 0})
    .orderBy(F.desc("inDegree"), F.desc("outDegree"), F.asc("name"))
)

degree_df.show(vertices.count(), truncate=False)


/home/hadoop/homework/.venv/lib/python3.10/site-packages/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")



[Stage 41:>                 (0 + 4) / 8][Stage 42:>                 (0 + 0) / 8]




[Stage 42:=====================>                                    (3 + 4) / 8]



+---+---------------------------------------------+--------+---------+
|id |name                                         |inDegree|outDegree|
+---+---------------------------------------------+--------+---------+
|70 |San Francisco Caltrain (Townsend at 4th)     |34810   |26304    |
|69 |San Francisco Caltrain 2 (330 Townsend)      |22523   |21758    |
|50 |Harry Bridges Plaza (Ferry Building)         |17810   |17255    |
|61 |2nd at Townsend                              |15463   |14026    |
|65 |Townsend at 7th                              |15422   |13752    |
|60 |Embarcadero at Sansome                       |15065   |14158    |
|77 |Market at Sansome                            |13916   |11431    |
|74 |Steuart at Market                            |13617   |13687    |
|55 |Temporary Transbay Terminal (Howard at Beale)|12966   |14436    |
|39 |Powell Street BART                           |10239   |9695     |
|67 |Market at 10th                               |10220   |11885    |
|76 |M

## Question 2

Find any two stations that have distance greater than 5 km.


In [5]:
long_distance_routes = (
    route_edges
    .filter(F.col("distance_km") > 5)
    .join(vertices.select(F.col("id").alias("src"), F.col("name").alias("src_name")), on="src")
    .join(vertices.select(F.col("id").alias("dst"), F.col("name").alias("dst_name")), on="dst")
    .select("src_name", "dst_name", "trip_count", "distance_km", "distance_m")
    .orderBy(F.desc("distance_km"), F.desc("trip_count"))
)

long_distance_routes.show(20, truncate=False)



[Stage 57:==============>                                           (2 + 4) / 8]



+----------------------------------------+----------------------------------------+----------+-----------+----------+
|src_name                                |dst_name                                |trip_count|distance_km|distance_m|
+----------------------------------------+----------------------------------------+----------+-----------+----------+
|MLK Library                             |Market at 4th                           |1         |67.85      |67850.4   |
|Castro Street and El Camino Real        |Howard at 2nd                           |2         |52.498     |52497.79  |
|Powell Street BART                      |San Antonio Shopping Center             |1         |50.168     |50168.41  |
|San Antonio Caltrain Station            |Market at 4th                           |2         |49.696     |49696.1   |
|San Francisco Caltrain (Townsend at 4th)|San Antonio Caltrain Station            |1         |48.331     |48330.74  |
|Park at Olive                           |San Francisco 

## Question 3

Find any two stations `A` and `C` that are connected by one hop `B` and have a total distance greater than `150`.

The homework statement does not specify a unit for `150`, so this notebook follows the text literally and applies the threshold to meters. If your instructor intended `15 km`, change the filter from `150` to `15000`.


In [6]:
one_hop_paths = (
    route_graph.find("(a)-[ab]->(b); (b)-[bc]->(c)")
    .filter("a.id <> c.id")
    .withColumn("total_distance_m", F.col("ab.distance_m") + F.col("bc.distance_m"))
    .withColumn("total_distance_km", F.round(F.col("total_distance_m") / F.lit(1000.0), 3))
    .filter(F.col("total_distance_m") > 150)
    .select(
        F.col("a.name").alias("station_a"),
        F.col("b.name").alias("station_b"),
        F.col("c.name").alias("station_c"),
        F.col("ab.trip_count").alias("a_to_b_trips"),
        F.col("bc.trip_count").alias("b_to_c_trips"),
        "total_distance_m",
        "total_distance_km",
    )
    .dropDuplicates(["station_a", "station_b", "station_c"])
    .orderBy(F.desc("total_distance_m"), F.desc("a_to_b_trips"), F.desc("b_to_c_trips"))
)

one_hop_paths.show(20, truncate=False)



[Stage 71:>                                                         (0 + 4) / 8]




[Stage 72:=====================>                                    (3 + 4) / 8]




[Stage 72:===========>      (5 + 3) / 8][Stage 74:>                 (0 + 1) / 1]



+----------------------------------------+----------------------------------------+----------------------------------------+------------+------------+-----------------+-----------------+
|station_a                               |station_b                               |station_c                               |a_to_b_trips|b_to_c_trips|total_distance_m |total_distance_km|
+----------------------------------------+----------------------------------------+----------------------------------------+------------+------------+-----------------+-----------------+
|MLK Library                             |Market at 4th                           |Stanford in Redwood City                |1           |1           |105733.31        |105.733          |
|San Francisco Caltrain (Townsend at 4th)|San Antonio Caltrain Station            |Market at 4th                           |1           |2           |98026.84         |98.027           |
|Park at Olive                           |San Francisco Caltrain 

## Question 4

Run PageRank to find the importance of all stations.


In [7]:
pagerank_df = (
    route_graph.pageRank(resetProbability=0.15, maxIter=10)
    .vertices
    .select("id", "pagerank")
    .join(vertices.select("id", "name"), on="id", how="left")
    .orderBy(F.desc("pagerank"), F.asc("name"))
)

pagerank_df.show(20, truncate=False)



[Stage 88:=============================>                            (4 + 4) / 8]




[Stage 96:>                                                         (0 + 1) / 1]



+---+------------------+----------------------------------------+
|id |pagerank          |name                                    |
+---+------------------+----------------------------------------+
|28 |1.1866376150732876|Mountain View Caltrain Station          |
|36 |1.1794135090384048|California Ave Caltrain Station         |
|70 |1.1587431716882888|San Francisco Caltrain (Townsend at 4th)|
|76 |1.128908090708634 |Market at 4th                           |
|9  |1.1043342659713267|Japantown                               |
|14 |1.1016277933945782|Arena Green / SAP Center                |
|80 |1.0986372320499118|Santa Clara County Civic Center         |
|38 |1.0980139649748932|Park at Olive                           |
|34 |1.0874935584266194|Palo Alto Caltrain Station              |
|63 |1.0865717465253095|Howard at 2nd                           |
|29 |1.0865639025291938|San Antonio Caltrain Station            |
|31 |1.0833417451328378|San Antonio Shopping Center             |
|3  |1.077

In [8]:
spark.stop()
